### Лабораторная работа №5 по ТМО

#### Ансамбли моделей машинного обучения. Часть 1.

Цель лабораторной работы: изучение ансамблей моделей машинного обучения.

Задание:

1. Выберите набор данных (датасет) для решения задачи классификации или регресии.
2. В случае необходимости проведите удаление или заполнение пропусков и кодирование категориальных признаков.
3. С использованием метода train_test_split разделите выборку на обучающую и тестовую.
4. Обучите следующие ансамблевые модели:
+ две модели группы бэггинга (бэггинг или случайный лес или сверхслучайные деревья);
+ AdaBoost;
+ градиентный бустинг.

5. Оцените качество моделей с помощью одной из подходящих для задачи метрик. Сравните качество полученных моделей.

В качестве исходного датасета возьму __California Housing Prices Dataset__ - https://www.kaggle.com/datasets/nalisha/california-housing-prices-dataset-clean-and-ml

Этот набор данных содержит подробную информацию о жилых кварталах Калифорнии, включая географическое расположение, статистику населения и уровень доходов. 

Каждая строка представляет собой конкретный район, а столбцы описывают различные характеристики, такие как медианный доход, количество комнат, численность населения и близость к океану.

В этом датасете следующие столбцы:

+ **longitude** - Географические координаты (положение восток-запад)
+ **latitude** - Географические координаты (положение с севера на юг)
+ **housing_median_age** - Средний возраст домов в этом районе
+ **total_rooms** - Общее количество комнат во всех домах
+ **total_bedrooms** - Общее количество спален
+ **population** - Общая численность населения в этом районе
+ **households** - Количество домохозяйств (семей)
+ **median_income** - Медианный доход жителей
+ **median_house_value** - Медианная цена на жилье (целевая переменная)
+ **ocean_proximity** - Расстояние от океана (категориальный признак)

Первым делом загрузим датасет и убедимся, что нет пропусков (при необходимости удалим пропуски), затем посмотрим на список столбцов и их типы.

In [1]:
# добавим необходимые модули
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [3]:
houses_origin = pd.read_csv("./housing.csv")

houses_origin.head(10)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
5,-122.25,37.85,52.0,919.0,213.0,413.0,193.0,4.0368,269700.0,NEAR BAY
6,-122.25,37.84,52.0,2535.0,489.0,1094.0,514.0,3.6591,299200.0,NEAR BAY
7,-122.25,37.84,52.0,3104.0,687.0,1157.0,647.0,3.1200,241400.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY
9,-122.25,37.84,52.0,3549.0,707.0,1551.0,714.0,3.6912,261100.0,NEAR BAY


In [4]:
print('Всего строк: {}'.format(houses_origin.shape[0]))

print(f"\nВсего пропусков: {houses_origin.isnull().sum().sum()}")

for i in list(houses_origin.columns):
    print(i + "\t" + str(houses_origin[i].isnull().sum()) + "\t" + str(round(houses_origin[i].isnull().mean()*100, 2)) + "%")

Всего строк: 20640

Всего пропусков: 207
longitude	0	0.0%
latitude	0	0.0%
housing_median_age	0	0.0%
total_rooms	0	0.0%
total_bedrooms	207	1.0%
population	0	0.0%
households	0	0.0%
median_income	0	0.0%
median_house_value	0	0.0%
ocean_proximity	0	0.0%


Видим 207 пропусков в наборе - в столбце "число спален" недобор. Ввиду малого числа пропусков (всего 1% - просто удалю соответствующие строки)

In [5]:
houses_cleaned = houses_origin.dropna()

print('Всего строк: {}'.format(houses_cleaned.shape[0]))

for i in list(houses_cleaned.columns):
    print(i + "\t" + str(houses_cleaned[i].isnull().sum()) + "\t" + str(round(houses_cleaned[i].isnull().mean()*100, 2)) + "%")

Всего строк: 20433
longitude	0	0.0%
latitude	0	0.0%
housing_median_age	0	0.0%
total_rooms	0	0.0%
total_bedrooms	0	0.0%
population	0	0.0%
households	0	0.0%
median_income	0	0.0%
median_house_value	0	0.0%
ocean_proximity	0	0.0%


Теперь порядок

In [6]:
print("Список колонок с типами данных: ")
houses_cleaned.dtypes

Список колонок с типами данных: 


longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity           str
dtype: object

В целом всё замечательно - данные однотипные, но ocean_proximity как категориальный признак надо бы перевести в числовой вид (чем меньше значение, тем ближе к океану)

In [8]:
# Смотрим уникальные значения и их частоту
print("Уникальные значения ocean_proximity:")
print(houses_cleaned['ocean_proximity'].unique())
print("\nРаспределение по категориям:")
print(houses_cleaned['ocean_proximity'].value_counts())

Уникальные значения ocean_proximity:
<StringArray>
['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']
Length: 5, dtype: str

Распределение по категориям:
ocean_proximity
<1H OCEAN     9034
INLAND        6496
NEAR OCEAN    2628
NEAR BAY      2270
ISLAND           5
Name: count, dtype: int64


По найденной информации:

<1H OCEAN - Менее 1 часа пути до океана

INLAND - Внутренние районы, далеко от побережья

NEAR OCEAN - Рядом с океаном, но не на берегу

NEAR BAY - Рядом с заливом (Сан-Франциско, Лос-Анджелес)

ISLAND - Островные территории (очень редко)

Ввиду малого числа, island просто уберём, а остальный закодируем порядково

In [ ]:
houses_categs = houses_cleaned[houses_cleaned['ocean_proximity'] != 'ISLAND']

# Задаём порядок: от "дальше от воды" к "ближе к воде/заливу" (значения приняты гипотетические)
order_mapping = {
    'INLAND': 0,           # в глубине материка
    'NEAR OCEAN': 1,       # рядом с океаном
    '<1H OCEAN': 2,        # в пределах часа от океана
    'NEAR BAY': 3          # у залива (часто самые дорогие районы)
}

# Применяем маппинг
houses_categs['ocean_proximity_ord'] = houses_categs['ocean_proximity'].map(order_mapping)

houses_categs = houses_categs.drop(columns=['ocean_proximity'])

print("Уникальные значения ocean_proximity:")
print(houses_categs['ocean_proximity_ord'].unique())

print("Список колонок с типами данных: ")
houses_categs.dtypes

Уникальные значения ocean_proximity:
[3 2 0 1]
Список колонок с типами данных: 


longitude              float64
latitude               float64
housing_median_age     float64
total_rooms            float64
total_bedrooms         float64
population             float64
households             float64
median_income          float64
median_house_value     float64
ocean_proximity_ord      int64
dtype: object

Данные обработаны. Дальше делим на обучающую и тестовую выборки, после чего обучим модели

In [14]:
# Удаляем исходный текстовый столбец, чтобы модели не сломались
X = houses_categs.drop(['median_house_value'], axis=1)
y = houses_categs['median_house_value']

from sklearn.model_selection import train_test_split

# X - признаки (уже с закодированным ocean_proximity)
# y - целевая переменная (median_house_value)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.25,      # 25% на тест, 75% на обучение
    random_state=42,     # фиксация для воспроизводимости
    shuffle=True
)
print(f"Train: {X_train.shape[0]} строк | Test: {X_test.shape[0]} строк")

Train: 15321 строк | Test: 5107 строк


In [ ]:
import time
from sklearn.ensemble import (
    BaggingRegressor,
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Инициализация моделей
models = {
    "Bagging": BaggingRegressor(
        estimator=DecisionTreeRegressor(),
        n_estimators=50,
        random_state=42,
        n_jobs=-1
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ),
    "AdaBoost": AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3), 
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.8,
        random_state=42
    )
}

# 2. Цикл обучения и оценки
results = []
for name, model in models.items():
    print(f"Обучаем: {name}...")
    t_start = time.time()
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    t_elapsed = round(time.time() - t_start, 3)
    
    # Метрики регрессии
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    
    results.append({
        "Модель": name,
        "RMSE ($)": round(rmse, 2),
        "MAE ($)": round(mae, 2),
        "R^2": round(r2, 4),
        "Время (с)": t_elapsed
    })

# 3. Вывод таблицы результатов
houses_results = pd.DataFrame(results)
print("\nИтоговая таблица качества на тестовой выборке:")
print(houses_results.to_markdown(index=False))

Обучаем: Bagging...


Обучаем: Random Forest...
Обучаем: AdaBoost...
Обучаем: Gradient Boosting...

Итоговая таблица качества на тестовой выборке:
| Модель            |   RMSE ($) |   MAE ($) |    R^2 |   Время (с) |
|:------------------|-----------:|----------:|-------:|------------:|
| Bagging           |    48887.4 |   31781.9 | 0.8188 |      11.59  |
| Random Forest     |    49092.2 |   32425.9 | 0.8173 |       1.703 |
| AdaBoost          |    78544.7 |   63872   | 0.5322 |       5.889 |
| Gradient Boosting |    55369   |   38506.3 | 0.7675 |       5.015 |


В итоге:

1. **Лидеры**: `Bagging` и `Random Forest` показали практически идентичную точность (`R:2 ≈ 0.82`). Это ожидаемо: оба метода относятся к бэггингу и работают за счёт усреднения множества независимых деревьев, что эффективно снижает дисперсию и стабилизирует прогноз на зашумлённых данных о ценах.

2. **Скорость**: Random Forest обучился в ~7 раз быстрее бэггинга благодаря оптимизированной реализации и параллельному построению деревьев.

3. **AdaBoost**: Показал наименьшее качество (`R^2 ~= 0.53`). Это связано с высокой чувствительностью алгоритма к выбросам: в целевой переменной `median_house_value` присутствуют районы с аномально высокими ценами, которые AdaBoost пытается "переучить", что ухудшает обобщающую способность. Для стабильной работы алгоритму обычно требуется более тщательный подбор `learning_rate` и глубины базовых деревьев.

4. **Gradient Boosting**: Несмотря на теоретическое преимущество в снижении смещения, без кросс-валидационной настройки гиперпараметров уступил бэггинговым методам. Тем не менее, показал адекватный результат (`R^2 ~= 0.77`).

**Итог**: Для табличных регрессионных задач с умеренным количеством признаков и наличием шума ансамбли на основе бэггинга (особенно Random Forest) часто являются оптимальным выбором по соотношению "качество/время обучения/устойчивость к настройкам".